In [0]:
# ============================================================
# GOLD LAYER - FACT_TRANSACTIONS - CORRECTED VERSION
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# GET ENVIRONMENT FROM ADF
# ============================================================

try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Using default: {env}")

# ============================================================
# STORAGE ACCOUNT
# ============================================================

storage_account_name = "capstonestorage01"

# ============================================================
# CONTAINER NAMES BASED ON ENVIRONMENT
# ============================================================

if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# ============================================================
# BUILD PATHS
# ============================================================

SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# LOAD DATA
# ============================================================

print("📖 Loading data...")

df_txn = spark.read.format("delta").load(f"{SILVER}/transactions")
dim_customer = spark.read.format("delta").load(f"{GOLD}/dim_customer")
dim_account = spark.read.format("delta").load(f"{GOLD}/dim_account")

print(f"✅ Transactions: {df_txn.count():,} rows")
print(f"✅ Dim Customer: {dim_customer.count():,} rows")
print(f"✅ Dim Account: {dim_account.count():,} rows")

# ============================================================
# JOIN & TRANSFORM - CREATE FACT_TRANSACTIONS
# ============================================================

print("\n🔄 Joining and transforming...")

fact_transactions = df_txn \
    .join(dim_customer.select(
        "customer_id", "risk_segment", "income_band", "age_group", "city"),
        "customer_id", "left") \
    .join(dim_account.select(
        "account_id", "credit_limit", "credit_tier", "rate_band", "interest_rate"),
        "account_id", "left") \
    .select(
        F.col("transaction_id"),
        F.col("transaction_date"),
        F.col("account_id"),
        F.col("customer_id"),
        F.col("product_type"),
        F.col("transaction_type"),
        F.col("transaction_status"),
        F.col("amount"),
        F.col("outstanding_balance"),
        F.col("exposure"),
        F.col("interest_accrued"),
        F.col("risk_segment"),
        F.col("income_band"),
        F.col("age_group"),
        F.col("city"),
        F.col("credit_limit"),
        F.col("credit_tier"),
        F.col("rate_band"),
        F.col("risk_band"),
        F.col("year"),
        F.col("month"),
        
        # Utilization percentage
        F.round(F.col("outstanding_balance") / F.col("credit_limit") * 100, 2)
            .alias("utilization_pct"),
        
        # Successful amount
        F.when(F.col("transaction_status") == "Success", F.col("amount"))
            .otherwise(F.lit(0.0)).alias("successful_amount"),
        
        # Repayment amount
        F.when(F.col("transaction_type") == "Repayment", F.col("amount"))
            .otherwise(F.lit(0.0)).alias("repayment_amount"),
        
        # Disbursement amount
        F.when(F.col("transaction_type") == "Disbursement", F.col("amount"))
            .otherwise(F.lit(0.0)).alias("disbursement_amount"),
        
        # Fee amount
        F.when(F.col("transaction_type") == "Fee", F.col("amount"))
            .otherwise(F.lit(0.0)).alias("fee_amount"),
        
        F.current_timestamp().alias("gold_created_at")
    )

print(f"🔄 Fact transactions rows: {fact_transactions.count():,}")

# ============================================================
# WRITE TO GOLD
# ============================================================

print("\n💾 Writing to Gold...")

fact_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("product_type", "year") \
    .save(f"{GOLD}/fact_transactions")

# ============================================================
# SUMMARY
# ============================================================

print(f"\n{'='*55}")
print(f"✅ fact_transactions : {fact_transactions.count():,} rows")
print(f"📁 Written to: {GOLD}/fact_transactions")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")